# Проверка динамик пары мёбиус-колец (3 ноя 2026)
Каждый раздел: вывод (Markdown) → численная проверка, печатает **OK/FAIL**.
Ядро SageMath 9.6 (код на numpy, работает и в Python 3).
Обозначения: кольцо радиуса $R$ в плоскости $z=sH$, $s=+1$ (N), $s=-1$ (S); базис $(e_r,e_\tau,e_z)$, $e_\tau$ — касательная к линии;
$\varphi=\omega t$; $\chi$ — азимут от особой точки (A, C) или от разреза (B); плоскость поворота $b$: $(u,v)\in\{(r,z),(r,\tau),(z,\tau)\}$.
$$b=s\,b_0(\cos\psi\,u+\sin\psi\,v),\quad \psi_A=\tfrac{\chi}{2}+\tfrac{\varphi}{2},\ \ \psi_B=-\tfrac{\chi}{2}+\tfrac{\varphi}{2},\ \ \psi_C=\tfrac{\chi}{2}+\varphi .$$

In [ ]:
import numpy as np
R, H, B0 = 4.0, 3.0, 1.0
TW = 2*np.pi
m = lambda x: np.mod(x, TW)
def fr(th):
    th = np.asarray(th, float)
    return {'r': np.stack([np.cos(th), np.sin(th), 0*th], -1),
            't': np.stack([-np.sin(th), np.cos(th), 0*th], -1),
            'z': np.stack([0*th, 0*th, 1+0*th], -1)}
PL = {'rz': ('r','z'), 'rt': ('r','t'), 'tz': ('t','z')}
def burgers(kind, s, th, phi, pl, alpha=None):
    # alpha — азимут особой точки при phi=0 (по умолчанию: N в 0, S в π — противофаза)
    if alpha is None: alpha = 0.0 if s > 0 else np.pi
    f = fr(th); u, v = PL[pl]
    if kind == 'A': p = m(th - phi - alpha)/2 + phi/2
    elif kind == 'B': p = -m(th - alpha)/2 + phi/2
    else: p = m(th - phi - alpha)/2 + phi
    return s*B0*(np.cos(p)[..., None]*f[u] + np.sin(p)[..., None]*f[v])
def report(name, ok): print(('OK   ' if ok else 'FAIL ') + name); return ok
TH = np.linspace(0.003, TW - 0.003, 721)

## 1. Период и статистика
Для A и B при $\varphi\to\varphi+2\pi$ получаем $\psi\to\psi+\pi$, т.е. $b\to-b$ (спинорная двузначность, статья: «Период пары и статистика»), возврат за $4\pi$.
Для C: $\psi\to\psi+2\pi$ за один оборот — бозон.

In [ ]:
for k in 'ABC':
    for pl in PL:
        b0 = burgers(k, 1, TH, .3, pl); b2 = burgers(k, 1, TH, .3+TW, pl); b4 = burgers(k, 1, TH, .3+2*TW, pl)
        exp2 = -b0 if k in 'AB' else b0
        report(f'{k} {pl}: φ+2π → {"−b" if k in "AB" else "+b"},  φ+4π → b', np.allclose(b2, exp2) and np.allclose(b4, b0))

## 2. Шов и непрерывная прецессия
A и C: скачок $|\Delta b|=2b_0$ на бегущей особой точке. B: в фиксированной точке $b$ вращается непрерывно с частотой $\omega/2$.

In [ ]:
e = 1e-7
for k in 'AC':
    j = burgers(k, 1, np.array([.3 - e + 0]), .3, 'rz')[0] - burgers(k, 1, np.array([.3 + e]), .3, 'rz')[0]
    report(f'{k}: |Δb| = 2b0 на особой точке', abs(np.linalg.norm(j) - 2*B0) < 1e-5)
ph = np.linspace(0, 2*TW, 801); th0 = 1.0
bb = np.array([burgers('B', 1, np.array([th0]), p, 'rz')[0] for p in ph]); f0 = fr(np.array([th0]))
ang = np.unwrap(np.arctan2(bb @ f0['z'][0], bb @ f0['r'][0]))
report('B: dψ/dφ = 1/2 в фиксированной точке', np.allclose(np.diff(ang), (ph[1]-ph[0])/2, atol=1e-9))

## 3. Суммарный вектор Бюргерса пары (контур вокруг обоих колец)
При сдвиге особой точки S на $\Delta$ относительно N сумма $b_N+b_S$ в одной азимутальной точке есть разность двух единичных векторов с углом между ними $\Delta/2$ или $\Delta/2-\pi$ (в зависимости от дуги), откуда
$$\max_\theta|b_N+b_S| = 2b_0\max\left(|\cos\tfrac{\Delta}{4}|,\ |\sin\tfrac{\Delta}{4}|\right)\ \ (\Delta\neq0),\qquad =0\ \ (\Delta=0).$$
При противофазе $\Delta=\pi$: $|b_N+b_S|=\sqrt2\,b_0$ **во всех точках кольца** — пара не нейтральна.

In [ ]:
phis = np.linspace(0, 2*TW, 25)
def maxsum(k, rings, pl='rz'):
    return max(np.abs(sum(burgers(k, s, TH, p, pl, a) for s, a in rings)).max() for p in phis)
for D in (45, 90, 180, 270):
    d = np.radians(D); num = maxsum('A', [(1, 0.), (-1, d)]); an = 2*B0*max(abs(np.cos(d/4)), abs(np.sin(d/4)))
    report(f'Δ={D}°: max|ΣB| = {num:.3f} = формула {an:.3f}', abs(num - an) < 2e-2)
report('Δ=0°: ΣB = 0', maxsum('A', [(1, 0.), (-1, 0.)]) < 1e-9)
for k in 'ABC':
    for pl in PL:
        report(f'{k} {pl}: противофаза → |ΣB| = √2·b0 всюду', abs(maxsum(k, [(1,0.),(-1,np.pi)], pl) - np.sqrt(2)*B0) < 1e-6)

## 4. Остаток не гасится одним дефектом Вольтерры (deWit 1973 II, ур. 5.5)
Классический дефект добавляет $a+\Omega\times x$ (гармоники по азимуту $n\le1$). Подгонка МНК остатка $S(\theta)$: если остаток после подгонки сравним с $|S|$, компенсация невозможна.

In [ ]:
def volterra_fit(S):
    X = R*fr(TH)['r']; A = np.zeros((3*len(TH), 6))
    for i, x in enumerate(X):
        A[3*i:3*i+3, :3] = np.eye(3)
        A[3*i:3*i+3, 3:] = np.array([[0, x[2], -x[1]], [-x[2], 0, x[0]], [x[1], -x[0], 0]])
    sol = np.linalg.lstsq(A, S.reshape(-1), rcond=None)[0]
    return np.abs((A@sol).reshape(-1, 3) - S).max()
for k in 'AB':
    for pl in PL:
        S = burgers(k, 1, TH, .3, pl) + burgers(k, -1, TH, .3, pl)
        r = volterra_fit(S)
        report(f'{k} {pl}: остаток после a+Ω×x = {r:.2f} b0 (не мал → одной линией не погасить)', r > 0.5*B0)

## 5. Нейтральность и поперечный диполь — теорема
$\sum_k b_k(\theta)=0$ при всех $\theta,\varphi$ ⇔ в каждой азимутальной позиции особых точек $\sum s_k=0$.
Но тогда поперечный дипольный момент особых точек $\sum_k s_k R\,e_r(\varphi+\alpha_k)=0$: остаётся только **осевой** (статический) момент.
⇒ нейтральная конфигурация не имеет вращающегося поперечного диполя (и дипольного излучения, ср. $\tau\sim10^{-18}$ с).
Три кольца нейтральными быть не могут ($\sum s_k$ при нечётном числе ≠ 0).

In [ ]:
import itertools
def dip(rings, zs, phi):
    return sum(s*np.array([R*np.cos(phi+a), R*np.sin(phi+a), z]) for (s, a), z in zip(rings, zs))
grid = np.radians(np.arange(0, 360, 45)); found = {3: 0, 4: 0}; transverse_ok = True
for N in (3, 4):
    for signs in itertools.product((1, -1), repeat=N):
        for al in itertools.product(grid, repeat=N):
            pos = {}
            for s, a in zip(signs, al): pos[round(np.degrees(a)) % 360] = pos.get(round(np.degrees(a)) % 360, 0) + s
            if all(v == 0 for v in pos.values()):
                found[N] += 1
                zs = [H*(1 - 0.1*i)*s for i, s in enumerate(signs)]
                transverse_ok &= np.allclose(dip(list(zip(signs, al)), zs, .7)[:2], 0)
report('3 кольца: нейтральных конфигураций нет', found[3] == 0)
report(f'4 кольца: нейтральные есть ({found[4]})', found[4] > 0)
report('у всех нейтральных поперечный диполь = 0 (остаётся только осевой)', transverse_ok)
cfg = [(1, 0.), (-1, 0.)]
report('пара с полюсами в фазе нейтральна во всех A/B/C и плоскостях',
       all(maxsum(k, cfg, pl) < 1e-9 for k in 'ABC' for pl in PL))

## 6. Нелинейная теория: мёбиус-кольцо как голономия конечной дисклинации
В линейной теории скачок $b+\Omega\times x$ малый и линейный по $x$; поворот на $\pi$ невозможен.
В нелинейной калибровочной теории $SO(3)\ltimes T(3)$ (Kadić–Edelen, Katanaev–Volovich) локальный $b$, измеренный в параллельно переносимом репере, получает **голономию** $Q\in SO(3)$ при обходе петли, сцепленной с дисклинацией.
Закон $\psi=\chi/2$ даёт $b_{lab}(\theta)=R_z(\theta)\,R_n(\theta/2)\,b_{loc}$; при $\theta=2\pi$: $Q=R_{n}(\pi)$ — конечный поворот на $\pi$:
- $(r,\tau)$: $n=e_z$ — **клиновая** π-дисклинация вдоль оси спина (как $k=\tfrac12$ у Kleman–Friedel);
- $(r,z)$ и $(z,\tau)$: $n\perp e_z$ — **кручения** (twist) π-дисклинация.

Проверяем: $b(2\pi^-)=Q\,b(0^+)$ с $Q$ — ортогональной матрицей поворота на $\pi$, ось которой ожидаемая.

In [ ]:
def rot(n, a):
    n = np.asarray(n, float)/np.linalg.norm(n); K = np.array([[0,-n[2],n[1]],[n[2],0,-n[0]],[-n[1],n[0],0]])
    return np.eye(3) + np.sin(a)*K + (1-np.cos(a))*K@K
expected_axis = {'rt': [0,0,1], 'rz': [0,1,0], 'tz': [1,0,0]}   # оси при θ=0: e_z, e_τ(0)=y, e_r(0)=x
for pl in PL:
    n = expected_axis[pl]; Q = rot(n, np.pi)
    # базис плоскости (u,v) и ось n образуют тройку; проверяем голономию на векторах плоскости
    ok = True
    for psi0 in np.linspace(0, TW, 7):
        f0 = fr(np.array([1e-9])); f1 = fr(np.array([TW-1e-9])); u, v = PL[pl]
        b_start = np.cos(psi0)*f0[u][0] + np.sin(psi0)*f0[v][0]
        b_end = np.cos(psi0+np.pi)*f1[u][0] + np.sin(psi0+np.pi)*f1[v][0]   # ψ вырос на π за обход
        ok &= np.allclose(b_end, Q@b_start, atol=1e-6)
    report(f'{pl}: голономия = поворот на π вокруг {n} ({"клиновая" if pl=="rt" else "кручения"} π-дисклинация)', ok)
report('Q = R(π) ортогональна, det = 1, Q² = I (двузначный лифт в SU(2) → спин ½)',
       np.allclose(rot([0,0,1],np.pi)@rot([0,0,1],np.pi), np.eye(3)) and np.isclose(np.linalg.det(rot([0,0,1],np.pi)), 1))

## 7. Законы переноса (Fressengeas–Taupin 2020, ур. 88, 93–94)
$\dot\kappa^p=\theta\times V_\theta$, $\ \dot\alpha=-\mathrm{curl}\,\dot e^p+(\dot\kappa^p)^T-\mathrm{tr}(\dot\kappa^p)I$.
Для линейной дисклинации $\theta=t\otimes\Omega\,\delta$: $\dot\kappa^p\propto (t\times V)\otimes\Omega$.
- **B**: π-дисклинация голономии неподвижна (ось спина, $V_\theta=0$) — вращение $b$ есть поворот репера, $\dot\kappa^p=0$: кинематически свободно, без рождения дислокаций.
- **A, C**: шов бежит со скоростью $V=\omega R\,e_\tau$; дисклинация шва (перемычка N–S, вдоль $e_z$) движется поперёк своей линии: $t\times V\neq0$ ⇒ поток дислокаций $|\dot\alpha|\sim\pi\,\omega R$ на единицу длины, с сопротивлением $B_\theta$ — диссипация.
- Движение дефекта **вдоль** собственной линии ($V\parallel t$) потока не даёт.

In [ ]:
def kappa_dot(t, Omega, V):
    return np.outer(np.cross(t, V), Omega)      # ∝ θ×V для θ = t⊗Ω
phi = .7; om = 1.0
V_seam = om*R*fr(np.array([phi]))['t'][0]
kA = kappa_dot([0,0,1], np.pi*np.array([0,1,0]), V_seam)       # перемычка ∥ z, бежит по азимуту
kB = kappa_dot([0,0,1], np.pi*np.array([0,0,1]), np.zeros(3))  # клиновая π-дисклинация на оси, неподвижна
kL = kappa_dot(V_seam/np.linalg.norm(V_seam), np.pi*np.array([0,0,1]), V_seam)  # движение вдоль своей линии
report(f'A/C: бегущий шов рождает поток, |κ̇ᵖ| = {np.linalg.norm(kA):.2f} = π·ωR = {np.pi*om*R:.2f}', np.isclose(np.linalg.norm(kA), np.pi*om*R))
report('B: неподвижная дисклинация голономии, κ̇ᵖ = 0', np.allclose(kB, 0))
report('движение вдоль собственной линии: κ̇ᵖ = 0', np.allclose(kL, 0))

## Итог
| результат | статус |
|---|---|
| A, B — период 4π; C — 2π | OK |
| противофаза ⇒ $|b_N+b_S|=\sqrt2 b_0$; формула $2b_0\max(|\cos\Delta/4|,|\sin\Delta/4|)$ | OK |
| одной линией Вольтерры не гасится | OK |
| нейтральность ⇔ попарное гашение полюсов ⇒ нет поперечного диполя; 3 кольца — никогда | OK |
| в нелинейной теории мёбиус-кольцо = петля, сцепленная с конечной π-дисклинацией (голономия) | OK (кинематика) |
| по законам переноса B свободна, A/C порождают поток дислокаций | OK (оценка) |